# Anggota 5 — Set Uji dan Evaluasi Retrieval

**Faris · Tugas Akhir Pemrosesan Bahasa Alami · QA factoid tanpa LLM**

Notebook ini memeriksa bukti untuk 20 pertanyaan, menjalankan retrieval TF-IDF, dan menyimpan lima kandidat teratas per soal untuk evaluasi jawaban Anggota 6.

Kontribusi Faris adalah empat pertanyaan (FARIS-001 sampai FARIS-004) beserta jawaban rujukan dan ID passage buktinya. Benchmark retrieval memakai set gabungan 20 soal.

Jalankan notebook dari folder `Anggota5_Evaluasi_Retrieval` dengan kernel Python 3.11+ dari lingkungan `.venv`, lalu pilih **Run All**. Petunjuk instalasi ada di [README](README.md).

**Sumber korpus:** Pedoman Akademik FILKOM 2020, Tata Tertib UTS/UAS 2016, dan Pedoman Penilaian PTIIK 2015.

## 1. Membaca korpus dan set uji

Korpus dan metadata berasal dari Anggota 1. Set uji berisi 11 pertanyaan dari Anggota 2–4, empat pertanyaan kontribusi Faris, dan lima pertanyaan pelengkap.

In [1]:
import json
import sys
from pathlib import Path
from statistics import mean

from IPython.display import Markdown, display

sys.path.insert(0, "..")

from Anggota2_Pertanyaan.question_processing import proses_pertanyaan
from Anggota3_Retrieval.retrieval import Retrieval

with open("../Anggota1_Korpus/data/dokumen.json", encoding="utf-8") as file:
    dokumen = json.load(file)
with open("../Anggota1_Korpus/data/passages.jsonl", encoding="utf-8") as file:
    passages = [json.loads(line) for line in file if line.strip()]
with open("data/pertanyaan_uji.json", encoding="utf-8") as file:
    pertanyaan = json.load(file)

## 2. Memeriksa ID dan kutipan bukti

Setiap soal memiliki ID unik serta satu atau lebih passage yang mendukung jawaban rujukan. Kutipan diperiksa pada konteks dan teks passage dengan mengabaikan perbedaan huruf besar dan kecil.

In [2]:
teks_passage = {
    p["id"]: f'{p.get("konteks", "")} {p["teks"]}'.casefold() for p in passages
}
id_pertanyaan = [item["id_pertanyaan"] for item in pertanyaan]
assert len(id_pertanyaan) == len(set(id_pertanyaan)), "ID pertanyaan harus unik"
assert len(pertanyaan) == 20, f"Set uji harus berisi 20 soal, ditemukan {len(pertanyaan)}"
for item in pertanyaan:
    id_soal = item["id_pertanyaan"]
    relevan = set(item["passage_id"])
    assert relevan and relevan <= teks_passage.keys(), f"{id_soal}: ID passage tidak valid"

    bukti_per_passage = item.get("bukti_per_passage", {})
    assert not bukti_per_passage or bukti_per_passage.keys() == relevan, (
        f"{id_soal}: bukti_per_passage harus mencakup semua ID rujukan"
    )
    for passage_id, kutipan in bukti_per_passage.items():
        assert kutipan.strip() and kutipan.casefold() in teks_passage[passage_id], (
            f"{id_soal}: kutipan tidak ditemukan di {passage_id}"
        )

    bukti = item["bukti"].casefold()
    assert bukti.strip() and any(bukti in teks_passage[pid] for pid in relevan), (
        f"{id_soal}: kutipan utama tidak cocok dengan korpus"
    )

## 3. Menjalankan retrieval

Pertanyaan diolah dengan modul Anggota 2. Retriever Anggota 3 memberi skor cosine atas vektor TF-IDF, membuang kandidat berskor nol, dan memakai ID passage untuk memecahkan skor seri. Peringkat bukti pertama dihitung dari seluruh hasil berskor positif. Lima kandidat pertama disimpan untuk evaluasi jawaban.

In [3]:
retrieval = Retrieval(passages, dokumen)
hasil_soal = []

for item in pertanyaan:
    query = proses_pertanyaan(item["pertanyaan"])
    seluruh_kandidat = retrieval.cari_passage(query, top_k=len(passages))
    relevan = set(item["passage_id"])
    peringkat = next(
        (candidate["rank"] for candidate in seluruh_kandidat
         if candidate["passage_id"] in relevan),
        None,
    )
    top5 = seluruh_kandidat[:5]
    id_top5 = {candidate["passage_id"] for candidate in top5}
    ditemukan = sorted(id_top5 & relevan)
    hasil_soal.append({
        "id_pertanyaan": item["id_pertanyaan"],
        "penanya": item["penanya"],
        "sumber_kontribusi": item["sumber_kontribusi"],
        "pertanyaan": item["pertanyaan"],
        "jawaban_rujukan": item["jawaban"],
        "passage_id_relevan": item["passage_id"],
        "bukti_rujukan": item["bukti"],
        "peringkat_bukti_pertama": peringkat,
        "passage_relevan_di_top5": ditemukan,
        "precision_at_5": len(ditemukan) / 5,
        "recall_at_5": len(ditemukan) / len(relevan),
        "kandidat_top5": top5,
    })

## 4. Menghitung metrik dan melihat hasil per soal

- **Hit@5** adalah proporsi soal dengan setidaknya satu bukti pada lima hasil pertama.
- **MRR** merata-ratakan kebalikan peringkat bukti pertama. Jika bukti absen, nilainya nol.
- **Precision@5 macro** merata-ratakan proporsi passage relevan dalam lima posisi per soal.
- **Recall@5 macro** merata-ratakan proporsi bukti tercatat yang ditemukan dalam lima posisi per soal.

In [4]:
jumlah_hit5 = sum(bool(baris["passage_relevan_di_top5"]) for baris in hasil_soal)
peringkat_bukti = [baris["peringkat_bukti_pertama"] for baris in hasil_soal]
ringkasan = {
    "jumlah_pertanyaan": len(pertanyaan),
    "jumlah_passage": len(passages),
    "hit_at_5": jumlah_hit5 / len(pertanyaan),
    "hit_at_5_jumlah": jumlah_hit5,
    "mrr": mean(1 / rank if rank is not None else 0 for rank in peringkat_bukti),
    "precision_at_5_macro": mean(baris["precision_at_5"] for baris in hasil_soal),
    "recall_at_5_macro": mean(baris["recall_at_5"] for baris in hasil_soal),
    "metode": "TF-IDF unigram dan cosine similarity dari Anggota 3",
    "catatan": "Peringkat dihitung pada kandidat dengan skor positif; urutan seri memakai passage_id.",
}

display(Markdown(
    f'**Hit@5:** {jumlah_hit5}/{len(pertanyaan)} ({ringkasan["hit_at_5"]:.1%}) · '
    f'**MRR:** {ringkasan["mrr"]:.3f} · '
    f'**Precision@5 macro:** {ringkasan["precision_at_5_macro"]:.3f} · '
    f'**Recall@5 macro:** {ringkasan["recall_at_5_macro"]:.3f}'
))

tabel = [
    "| ID | Pertanyaan | Peringkat bukti | Bukti di top 5 |",
    "| --- | --- | ---: | --- |",
]
for baris in hasil_soal:
    peringkat = baris["peringkat_bukti_pertama"] or "—"
    bukti = ", ".join(baris["passage_relevan_di_top5"]) or "—"
    teks = baris["pertanyaan"].replace("|", r"\|")
    tabel.append(f'| {baris["id_pertanyaan"]} | {teks} | {peringkat} | {bukti} |')
display(Markdown("\n".join(tabel)))

**Hit@5:** 17/20 (85.0%) · **MRR:** 0.780 · **Precision@5 macro:** 0.200 · **Recall@5 macro:** 0.808

| ID | Pertanyaan | Peringkat bukti | Bukti di top 5 |
| --- | --- | ---: | --- |
| A2-001 | Berapa sks minimal untuk menyelesaikan studi sarjana? | 3 | pa2020-II-I-7 |
| A2-002 | Berapa lama waktu maksimal untuk Ujian Skripsi? | 1 | pa2020-VI-D-10 |
| A2-003 | Apa itu skripsi? | 19 | — |
| A2-004 | Apa nilai huruf untuk nilai angka 78? | 6 | — |
| A3-001 | Berapa lama waktu maksimal untuk menyelesaikan studi sarjana? | 2 | pa2020-II-I-8 |
| A3-002 | Kapan batas pengajuan cuti akademik setelah daftar ulang? | 1 | pa2020-III-G-2 |
| A3-003 | Apa rumus indeks prestasi semester? | 25 | — |
| A4-001 | Berapa persentase kehadiran minimal agar mahasiswa boleh mengikuti UAS? | 1 | pa2020-II-F-9, pa2020-III-K-1 |
| A4-002 | Berapa menit batas maksimal keterlambatan mengikuti ujian? | 1 | tatatertib-X-S1-3 |
| A4-003 | Berapa bobot nilai huruf B+? | 1 | pa2020-bobot-2, penilaian-bobot-2 |
| A4-004 | Siapa yang menetapkan pengecualian Dosen Pembimbing Skripsi? | 2 | pa2020-VI-A-11 |
| FARIS-001 | Berapa semester kumulatif yang harus ditempuh untuk evaluasi keberhasilan studi akhir tahun pertama? | 1 | pa2020-II-I-1, pa2020-II-I-3 |
| FARIS-002 | Berapa lama maksimal cuti akademik dan/atau terminal kuliah kumulatif untuk mahasiswa sarjana? | 1 | pa2020-III-G-4 |
| FARIS-003 | Berapa sks minimal yang harus dikumpulkan untuk menempuh ujian skripsi? | 1 | pa2020-VI-D-4 |
| FARIS-004 | Berapa hari kalender maksimal untuk menyelesaikan revisi skripsi sejak tanggal kelulusan ujian skripsi? | 1 | pa2020-VI-D-13 |
| SUP-001 | Kapan ujian susulan dilaksanakan paling lambat setelah UTS atau UAS? | 1 | tatatertib-X-S1-19 |
| SUP-002 | Berapa SKS lulus minimal yang harus dikumpulkan untuk dapat melanjutkan setelah evaluasi tahun kedua? | 1 | pa2020-II-I-4 |
| SUP-003 | Apa nilai huruf tertinggi yang dapat diberikan pada ujian khusus? | 1 | pa2020-II-F-4 |
| SUP-004 | Berapa lama masa percobaan mahasiswa alih program? | 1 | pa2020-III-H-7 |
| SUP-005 | Berapa sks beban studi maksimum untuk mahasiswa dengan IP semester sebelumnya minimal 3,00? | 1 | pa2020-tabel1-1 |

## 5. Analisis hasil di luar lima besar

Definisi skripsi bersaing dengan passage prosedural yang memakai kata “skripsi”. Soal nilai 78 bergantung pada kecocokan dengan rentang pada tabel konversi. Soal rumus IP bersaing dengan tabel beban studi yang berulang kali menyebut “IP” dan “semester”.

## 6. Menyimpan hasil untuk Anggota 6

Keluaran menyimpan `id_pertanyaan`, jawaban rujukan, peringkat bukti, dan `kandidat_top5` dalam format yang diterima ekstraktor Anggota 4.

In [5]:
hasil = {"ringkasan": ringkasan, "hasil_per_pertanyaan": hasil_soal}
path_hasil = Path("output/hasil_retrieval.json")
path_hasil.parent.mkdir(exist_ok=True)
with path_hasil.open("w", encoding="utf-8") as file:
    json.dump(hasil, file, ensure_ascii=False, indent=2)
    file.write("\n")

## 7. Contoh jawaban bot

Contoh ini memakai ekstraktor Anggota 4 untuk menjawab salah satu pertanyaan Faris berdasarkan lima kandidat hasil retrieval. Jawaban rujukan ditampilkan sebagai pembanding. Ganti `id_contoh` untuk melihat soal lain. Model UDPipe Indonesia diunduh pada pemakaian pertama.

In [6]:
from Anggota4.ekstraksi import EkstraktorJawaban

id_contoh = "FARIS-003"
contoh = next(baris for baris in hasil_soal if baris["id_pertanyaan"] == id_contoh)
jawaban_bot = EkstraktorJawaban(dokumen).ekstrak(
    proses_pertanyaan(contoh["pertanyaan"]), contoh["kandidat_top5"]
)

bukti_rujukan = ", ".join(contoh["passage_id_relevan"])

display(Markdown(f'''
**Pertanyaan:** {contoh["pertanyaan"]}

**Jawaban bot:** {jawaban_bot["jawaban"] or "Tidak ditemukan"}

**Jawaban rujukan:** {contoh["jawaban_rujukan"]}

**Bukti rujukan:** `{bukti_rujukan}` · Peringkat pertama: {contoh["peringkat_bukti_pertama"] or "Tidak ditemukan"}

**Sumber bot:** {jawaban_bot["sumber"] or "—"} (`{jawaban_bot["passage_id"] or "—"}`)
'''))


**Pertanyaan:** Berapa sks minimal yang harus dikumpulkan untuk menempuh ujian skripsi?

**Jawaban bot:** 120 sks

**Jawaban rujukan:** 138 sks

**Bukti rujukan:** `pa2020-VI-D-4` · Peringkat pertama: 1

**Sumber bot:** Buku Pedoman Akademik Fakultas Ilmu Komputer Universitas Brawijaya (2020), BAB VI A, hlm. 41 (`pa2020-VI-A-3`)


## 8. Kesimpulan dan batasan

Metrik mengukur penemuan passage relevan berdasarkan ID bukti yang tercatat pada set uji. Anggota 6 menggunakan lima kandidat tersebut untuk mengukur ketepatan jawaban akhir. Cakupan fakta mengikuti edisi dokumen dalam korpus.

Untuk setiap baris pada [hasil retrieval](output/hasil_retrieval.json), jalankan `proses_pertanyaan` lalu teruskan keluarannya dan `kandidat_top5` ke `EkstraktorJawaban.ekstrak`. Cocokkan hasil dengan jawaban rujukan menggunakan `id_pertanyaan`.